# UK Used Car Price Prediction: 01. Data Audit dan Cleaning

Notebook ini mengaudit [100,000 UK Used Car Data set](https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes) dan memanggil jalur pembersihan yang sama dengan CLI.

**Output:** `data/processed/cars_clean.csv` untuk notebook berikutnya dan `reports/data_audit.json` untuk bukti audit.

**Cara menjalankan:** letakkan CSV di `data/raw/`, pilih kernel `.venv`, lalu jalankan semua sel dari folder root atau `notebooks/`. File `cclass.csv` dan `focus.csv` opsional untuk pemeriksaan overlap.

## 1. Setup dan Load Data

In [1]:
import json
import sys
from hashlib import file_digest
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import BRANDS, load_raw_data, normalize_columns, prepare_data

pd.set_option("display.max_columns", None)
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_CLEAN = PROJECT_ROOT / "data" / "processed" / "cars_clean.csv"
AUDIT_REPORT = PROJECT_ROOT / "reports" / "data_audit.json"

In [2]:
for f in sorted(DATA_RAW.glob("*.csv")):
    tmp = pd.read_csv(f)
    print(f"{f.name:22} {str(tmp.shape):12} {list(tmp.columns)}")

audi.csv               (10668, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
bmw.csv                (10781, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
cclass.csv             (3899, 7)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'engineSize']
focus.csv              (5454, 7)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'engineSize']
ford.csv               (17965, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
hyundi.csv             (4860, 9)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax(£)', 'mpg', 'engineSize']
merc.csv               (13119, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
skoda.csv              (6267, 9)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg',

In [3]:
df = load_raw_data(DATA_RAW)
print(df.shape)
print(df.columns.tolist())
df.head()

(99187, 10)
['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'brand']


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,brand
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,audi


## 2. Audit Kualitas Data

In [4]:
df.info()
print("\nNilai kosong per kolom:")
print(df.isna().sum())
print("\nBaris duplikat:", df.duplicated().sum())
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 99187 entries, 0 to 99186
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         99187 non-null  str    
 1   year          99187 non-null  int64  
 2   price         99187 non-null  int64  
 3   transmission  99187 non-null  str    
 4   mileage       99187 non-null  int64  
 5   fuelType      99187 non-null  str    
 6   tax           99187 non-null  int64  
 7   mpg           99187 non-null  float64
 8   engineSize    99187 non-null  float64
 9   brand         99187 non-null  str    
dtypes: float64(2), int64(4), str(4)
memory usage: 7.6 MB

Nilai kosong per kolom:
model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
brand           0
dtype: int64

Baris duplikat: 1475


,year,price,mileage,tax,mpg,engineSize
count,99187.000000,99187.000000,99187.000000,99187.000000,99187.000000,99187.000000
mean,2017.087723,16805.347656,23058.914213,120.299838,55.166825,1.663280
std,2.123934,9866.773417,21148.523721,63.150926,16.138522,0.557646
min,1970.000000,450.000000,1.000000,0.000000,0.300000,0.000000
25%,2016.000000,9999.000000,7425.000000,125.000000,47.100000,1.200000
50%,2017.000000,14495.000000,17460.000000,145.000000,54.300000,1.600000
75%,2019.000000,20870.000000,32339.000000,145.000000,62.800000,2.000000
max,2060.000000,159999.000000,323000.000000,580.000000,470.800000,6.600000


In [5]:
print("year min/max      :", df["year"].min(), "/", df["year"].max())
print("engineSize == 0   :", (df["engineSize"] == 0).sum())
print("price min/max     :", df["price"].min(), "/", df["price"].max())
print("mileage min/max   :", df["mileage"].min(), "/", df["mileage"].max())
print()
for col in ["transmission", "fuelType", "brand"]:
    print(df[col].value_counts(dropna=False), "\n")

year min/max      : 1970 / 2060
engineSize == 0   : 273
price min/max     : 450 / 159999
mileage min/max   : 1 / 323000

transmission
Manual       56445
Semi-Auto    22677
Automatic    20056
Other            9
Name: count, dtype: int64 

fuelType
Petrol      54928
Diesel      40928
Hybrid       3078
Other         247
Electric        6
Name: count, dtype: int64 

brand
ford        17965
vw          15157
vauxhall    13632
merc        13119
bmw         10781
audi        10668
toyota       6738
skoda        6267
hyundi       4860
Name: count, dtype: int64 



## 3. Inspeksi Anomali

In [6]:
overlap_key = ["model", "year", "price", "transmission", "mileage", "fuelType", "engineSize"]
overlap_checks = {}

for name, brand in (("cclass.csv", "merc"), ("focus.csv", "ford")):
    path = DATA_RAW / name
    check = {"brand": brand}
    if path.is_file():
        subset = normalize_columns(pd.read_csv(path))
        main_keys = df.loc[df["brand"] == brand, overlap_key].drop_duplicates()
        matches = subset.merge(main_keys, on=overlap_key, how="left", indicator=True)
        matched = int(matches["_merge"].eq("both").sum())
        check.update(
            status="checked",
            rows=len(subset),
            matched_rows=matched,
            matched_fraction=matched / len(subset) if len(subset) else None,
        )
    else:
        check.update(status="skipped", reason="Optional CSV not found")
    overlap_checks[name] = check

print(json.dumps(overlap_checks, indent=2))

{
  "cclass.csv": {
    "brand": "merc",
    "status": "checked",
    "rows": 3899,
    "matched_rows": 3634,
    "matched_fraction": 0.932033854834573
  },
  "focus.csv": {
    "brand": "ford",
    "status": "checked",
    "rows": 5454,
    "matched_rows": 4670,
    "matched_fraction": 0.8562522918958563
  }
}


In [7]:
# 1. Tahun tidak wajar
print("year > 2020:", (df["year"] > 2020).sum())
print(df.loc[df["year"] > 2020, ["brand", "model", "year", "price", "mileage"]].head(10))
print("\nyear < 2000:", (df["year"] < 2000).sum())
print(df.loc[df["year"] < 2000, ["brand", "model", "year", "price", "mileage"]]
        .sort_values("year").head(10))

# 2. engineSize == 0
zero_eng = df[df["engineSize"] == 0]
print("\nengineSize == 0 per fuelType:\n", zero_eng["fuelType"].value_counts())
print("\nengineSize == 0 per brand:\n", zero_eng["brand"].value_counts())

# 3. mpg ekstrem
print("mpg > 100:", (df["mpg"] > 100).sum())
print(df.loc[df["mpg"] > 100, ["brand", "model", "year", "fuelType", "mpg"]]
        .head(10).to_string())
print("\nmpg < 10:", (df["mpg"] < 10).sum())
print(df.loc[df["mpg"] < 10, ["brand", "model", "year", "fuelType", "mpg"]]
        .head(10).to_string())

# 4. tax == 0
print("\ntax == 0:", (df["tax"] == 0).sum())
print(df.loc[df["tax"] == 0, "fuelType"].value_counts().to_string())

year > 2020: 1
      brand   model  year  price  mileage
39175  ford  Fiesta  2060   6495    54807

year < 2000: 22
          brand     model  year  price  mileage
81235  vauxhall    Zafira  1970  10495    37357
56346      merc   M Class  1970  24999    14000
20293       bmw  5 Series  1996   5995    36000
38327      ford    Escort  1996   3000    50000
56186      merc  SL CLASS  1997   9995    76000
76255  vauxhall     Astra  1997    950    89233
20509       bmw        Z3  1997   3950    49000
10427      audi        A8  1997   4650   122000
56082      merc       SLK  1998   1990   113557
56068      merc   S Class  1998  19990    43534

engineSize == 0 per fuelType:
 fuelType
Petrol      163
Diesel       69
Hybrid       38
Electric      2
Other         1
Name: count, dtype: int64

engineSize == 0 per brand:
 brand
audi        57
ford        51
bmw         47
hyundi      47
vauxhall    33
vw          15
merc        12
toyota       6
skoda        5
Name: count, dtype: int64
mpg > 100: 57

## 4. Keputusan pembersihan

| Keputusan | Alasan / asumsi |
|---|---|
| Pakai 9 file merek utama | File `cclass.csv` dan `focus.csv` merupakan subset dengan potensi overlap. Dua file `unclean` adalah versi mentah tambahan dan tidak digabung. |
| Seragamkan `SL CLASS` dan `X-CLASS` sebelum deduplikasi | Variasi ejaan tidak membuat kategori atau duplikat baru. |
| Hapus baris identik | Dataset tidak memiliki ID listing. Deduplikasi mempertahankan kemunculan pertama. |
| Pertahankan `year` 1995–2020 | Batas snapshot historis dan penanganan tahun yang tidak wajar, termasuk 1970 dan 2060. |
| Buang `engineSize=0` pada mobil non-listrik | Nol dianggap data mesin yang tidak tersedia untuk mobil non-listrik. |
| Buang `mpg < 10` | Nilai ekstrem rendah diduga berasal dari masalah input atau satuan. |
| Pertahankan `mpg > 100` dan `tax=0` | Nilai ini mungkin sah, misalnya untuk kendaraan plug-in hybrid atau kendaraan berpajak nol. |

`price` dan `tax` menggunakan GBP, `mileage` menggunakan mil, `mpg` menggunakan mil per galon, `engineSize` menggunakan liter, dan `year` adalah tahun model. Hash SHA-256 input dan bukti overlap disimpan di `reports/data_audit.json`.

In [8]:
# Jalur yang sama dengan python -m src.data.
df_clean = prepare_data(DATA_RAW, DATA_CLEAN)
removed = len(df) - len(df_clean)
print(f"Awal: {len(df):,} | Dibuang: {removed:,} | Bersih: {len(df_clean):,}")

Awal: 99,187 | Dibuang: 1,776 | Bersih: 97,411


## 5. Pemeriksaan kategori model

In [9]:
counts = df_clean.groupby(["brand", "model"]).size()
rare = counts[counts < 10]
print("Kombinasi brand-model:", len(counts))
print(f"Kombinasi dengan <10 listing: {len(rare)} ({rare.sum()} baris)")

Kombinasi brand-model: 194
Kombinasi dengan <10 listing: 36 (133 baris)


## 6. Simpan bukti audit

In [10]:
files = [f"{brand}.csv" for brand in BRANDS]
files.extend(name for name, check in overlap_checks.items() if check["status"] == "checked")
hashes = {}
for name in files:
    with (DATA_RAW / name).open("rb") as handle:
        hashes[name] = file_digest(handle, "sha256").hexdigest()

audit = {
    "dataset": "https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes",
    "input_sha256": hashes,
    "raw_rows": len(df),
    "clean_rows": len(df_clean),
    "removed_rows": removed,
    "overlap_key": overlap_key,
    "overlap_checks": overlap_checks,
}
AUDIT_REPORT.parent.mkdir(parents=True, exist_ok=True)
AUDIT_REPORT.write_text(json.dumps(audit, indent=2) + "\n", encoding="utf-8")
print(f"Audit tersimpan: {AUDIT_REPORT.relative_to(PROJECT_ROOT)}")

Audit tersimpan: reports\data_audit.json
